In [ ]:
import json
import uuid
import re
import pandas as pd

from snowflake.connector.pandas_tools import write_pandas
from snowflake.snowpark.context import get_active_session
from packaging.version import Version


s = get_active_session()

def info(msg):    print(f"INFO:    {msg}")
def warning(msg): print(f"WARNING: {msg}")
def error(msg):   print(f"ERROR:   {msg}")

### Set Doc Type Config

In [ ]:
staged_files = s.sql("""
    SELECT RELATIVE_PATH
    FROM DIRECTORY(@PERMAFROST_POC.INGEST.RAW_DOCUMENTS_STAGE)
    WHERE RELATIVE_PATH LIKE 'config/%.json'
""").collect()

all_files = [r["RELATIVE_PATH"].split("/")[-1] for r in staged_files]
info(f"Found {len(all_files)} config file(s): {all_files}")

if not all_files:
    warning("No config JSON files found in stage. Upload first")
else:
    # Group files by doc_type, pick highest version per type
    # Expects filenames like: health_certificate_1.3.json
    version_pattern = re.compile(r'^(.+?)_(\d+\.\d+)\.json$')

    latest_files = {}
    unversioned_files = []

    for filename in all_files:
        match = version_pattern.match(filename)
        if match:
            doc_type = match.group(1)   # e.g. health_certificate
            version  = match.group(2)   # e.g. 1.3
            if doc_type not in latest_files:
                latest_files[doc_type] = (version, filename)
            else:
                current_version = latest_files[doc_type][0]
                if Version(version) > Version(current_version):
                    latest_files[doc_type] = (version, filename)
        else:
            unversioned_files.append(filename)

    if unversioned_files:
        warning(f"Skipping unversioned files: {unversioned_files}")

    doc_type_files = [filename for _, filename in latest_files.values()]

    for doc_type, (version, filename) in latest_files.items():
        info(f"  [{doc_type}] using version {version} → {filename}")

    info(f"\n{len(doc_type_files)} latest config(s) selected: {doc_type_files}")

In [ ]:
processed_doc_types = []

for filename in doc_type_files:
    try:
        raw = s.sql(f"""
            SELECT $1::VARCHAR AS JSON_TEXT
            FROM @PERMAFROST_POC.INGEST.RAW_DOCUMENTS_STAGE/config/{filename}
            (FILE_FORMAT => PERMAFROST_POC.CONFIG.JSON_RAW_FORMAT)
        """).collect()

        json_text  = "\n".join([row["JSON_TEXT"] for row in raw])
        doc_config = json.loads(json_text)
        doc_type   = doc_config["doc_type"]

        display_name     = doc_config.get("display_name", doc_type.replace("_", " ").title())
        schema_version   = doc_config.get("schema_version", "1.0")
        mandatory_fields = doc_config.get("mandatory_fields", [])
        optional_fields  = doc_config.get("optional_fields", [])
        synonyms         = doc_config.get("synonyms", {})
        thresholds       = doc_config.get("confidence_thresholds", {
            "auto_approve": 0.85,
            "review_below": 0.85
        })
        weights          = doc_config.get("confidence_weights", {
            "field_completeness":  0.50,
            "extract_certainty":   0.50
        })
        dedup_key_fields = doc_config.get("dedup_key_fields", None)
        prompt_template  = doc_config.get("prompt_template", None)
        llm_model        = doc_config.get("llm_model", None)

        def sql_json_or_null(val):
            if val is None:
                return "NULL"
            return f"PARSE_JSON($${json.dumps(val)}$$)"

        def sql_str_or_null(val):
            if val is None:
                return "NULL"
            escaped = val.replace("'", "''")
            return f"'{escaped}'"

        merge_result = s.sql(f"""
            MERGE INTO PERMAFROST_POC.CONFIG.DOC_TYPE_CONFIG AS target
            USING (
                SELECT
                    '{doc_type}'                                          AS DOC_TYPE,
                    '{display_name}'                                      AS DOC_DISPLAY_NAME,
                    '{schema_version}'                                    AS SCHEMA_VERSION,
                    PARSE_JSON($${json.dumps(mandatory_fields)}$$)        AS MANDATORY_FIELDS,
                    PARSE_JSON($${json.dumps(optional_fields)}$$)         AS OPTIONAL_FIELDS,
                    {sql_json_or_null(synonyms if synonyms else None)}    AS SYNONYMS,
                    PARSE_JSON($${json.dumps(thresholds)}$$)              AS CONFIDENCE_THRESHOLDS,
                    PARSE_JSON($${json.dumps(weights)}$$)                 AS CONFIDENCE_WEIGHTS,
                    {sql_json_or_null(dedup_key_fields)}                  AS DEDUP_KEY_FIELDS,
                    {sql_str_or_null(prompt_template)}                    AS PROMPT_TEMPLATE,
                    {sql_str_or_null(llm_model)}                          AS LLM_MODEL,
                    TRUE                                                  AS IS_ACTIVE
            ) AS source
                ON  target.DOC_TYPE       = source.DOC_TYPE
                AND target.SCHEMA_VERSION = source.SCHEMA_VERSION
            WHEN MATCHED THEN UPDATE SET
                DOC_DISPLAY_NAME      = source.DOC_DISPLAY_NAME,
                MANDATORY_FIELDS      = source.MANDATORY_FIELDS,
                OPTIONAL_FIELDS       = source.OPTIONAL_FIELDS,
                SYNONYMS              = source.SYNONYMS,
                CONFIDENCE_THRESHOLDS = source.CONFIDENCE_THRESHOLDS,
                CONFIDENCE_WEIGHTS    = source.CONFIDENCE_WEIGHTS,
                DEDUP_KEY_FIELDS      = source.DEDUP_KEY_FIELDS,
                PROMPT_TEMPLATE       = source.PROMPT_TEMPLATE,
                LLM_MODEL             = source.LLM_MODEL,
                IS_ACTIVE             = source.IS_ACTIVE,
                UPDATED_AT            = CURRENT_TIMESTAMP()
            WHEN NOT MATCHED THEN INSERT (
                DOC_TYPE, DOC_DISPLAY_NAME, SCHEMA_VERSION,
                MANDATORY_FIELDS, OPTIONAL_FIELDS, SYNONYMS,
                CONFIDENCE_THRESHOLDS, CONFIDENCE_WEIGHTS, DEDUP_KEY_FIELDS,
                PROMPT_TEMPLATE, LLM_MODEL, IS_ACTIVE
            ) VALUES (
                source.DOC_TYPE, source.DOC_DISPLAY_NAME, source.SCHEMA_VERSION,
                source.MANDATORY_FIELDS, source.OPTIONAL_FIELDS, source.SYNONYMS,
                source.CONFIDENCE_THRESHOLDS, source.CONFIDENCE_WEIGHTS,
                source.DEDUP_KEY_FIELDS, source.PROMPT_TEMPLATE,
                source.LLM_MODEL, source.IS_ACTIVE
            )
        """).collect()

        rows_inserted = merge_result[0]["number of rows inserted"]
        rows_updated  = merge_result[0]["number of rows updated"]

        if rows_inserted > 0:
            s.sql(f"""
                UPDATE PERMAFROST_POC.CONFIG.DOC_TYPE_CONFIG
                SET    IS_ACTIVE  = FALSE,
                       UPDATED_AT = CURRENT_TIMESTAMP()
                WHERE  DOC_TYPE      = '{doc_type}'
                  AND  SCHEMA_VERSION != '{schema_version}'
                  AND  IS_ACTIVE      = TRUE
            """).collect()
            info(f"  New version {schema_version} inserted — previous versions deactivated")

        elif rows_updated > 0:
            info(f"  Version {schema_version} already exists — updated in place")

        info(f"Upserted DOC_TYPE_CONFIG: {doc_type} v{schema_version}")
        processed_doc_types.append((filename, doc_type, doc_config))

    except Exception as e:
        error(f"Failed to process {filename}: {e}")

### Set Pipeline Config

In [ ]:
CLASSIFY_PROMPT = """You are analyzing a multi-page file that may contain multiple documents.
Your job is to identify each distinct document — by both its type AND its boundaries.

Document types:
- health_certificate
- commercial_invoice
- packing_list
- bill_of_lading
- catch_certificate
- country_of_origin_certificate
- unknown

Rules:
- Each distinct physical document = one segment
- Look for new document headers, certificate numbers, reference numbers, issuing authority names
- A continuation page (e.g. page 2 of a long invoice) belongs to the same segment
- If uncertain whether two same-type pages are one doc or two, prefer splitting

IMPORTANT DISTINCTIONS:
- commercial_invoice: issued by the SELLER/SUPPLIER of seafood goods to 
  Slade Gorton for payment of the product itself. Contains product 
  descriptions, unit prices per lb/kg, and total product value.
- freight_invoice / shipping_invoice: issued by a CARRIER or FREIGHT 
  FORWARDE for transportation services. Contains charges for ocean freight, terminal handling, 
  surcharges, and logistics fees — NOT product prices. Classify as 
  'unknown' — do not classify as commercial_invoice.
- bill_of_lading: issued by the carrier as a transport contract and 
  receipt of goods. Different from a freight invoice which is a bill 
  for services.

For each segment also extract a one-sentence description containing:
- Supplier or exporter name (whoever issued or sent the document)
- Product description (species or commodity, description of goods)
- Document reference number:
  * health_certificate → certificate number
  * catch_certificate → certificate number  
  * commercial_invoice → invoice number
  * packing_list → packing list number
  * bill_of_lading → B/L number
  * country_of_origin_certificate → certificate number
- Issue date or the date representing when the document is created

Format: "[Supplier] — [Product] — [Doc Type Ref]: [Number] — [Date]"
Example: "PT. INDOKOM SAMUDRA PERSADA — FROZEN SHRIMPS — Cert No: 24.0-S-00629-2026 — 2026-03-15"
If any element is not found, omit it gracefully.

Return ONLY valid JSON, no explanation, no markdown:
{{
  "segments": [
    {{
      "doc_type": "<doc_type>",
      "page_start": <1-based int>,
      "page_end": <1-based int>,
      "confidence": <float 0.0-1.0>,
      "boundary_signal": "<brief reason why a new segment starts here>",
      "document_description": "<one sentence description>"
    }}
  ]
}}

Document pages:
{pages}"""

HAIKU_PROMPT = """Classify this document page as ENGLISH or NON_ENGLISH.

If the page contains a non-English language, classify as NON_ENGLISH.
Only classify as ENGLISH if every descriptive value you found is in English.
When uncertain, choose NON_ENGLISH.

If you are not confident the page is purely ENGLISH, return NON_ENGLISH.

Document page:
{page_text}

Return ONLY valid JSON:
{{"label": "ENGLISH" or "NON_ENGLISH", "reason": "one example value that determined the classification"}}"""

SONNET_LANG_PROMPT = """Classify this document page as ENGLISH, NON_ENGLISH, or BILINGUAL.

IGNORE these — they have no language: names, places, codes (HKP, COD, FIL), scientific names, numbers, dates, seals/stamps.
IGNORE column headers and field labels — only look at the DATA VALUES that fill those fields.

CLASSIFICATION RULES:
1. Default to NON_ENGLISH if values contain any non-English language.
2. Override to BILINGUAL only if you can find a SPECIFIC value that appears TWICE on the page — once in English and once in another language. Example: "Cod headed gutted / Треска обезглавленная потрошёная". A column header like "Product" above a Spanish value "Filet sin piel" does NOT count — that is a label, not a translated value.
3. Classify as ENGLISH only if all values (after ignoring names/codes/numbers) are English.

BILINGUAL TEST: To claim BILINGUAL, you must cite one value that has both an English AND non-English version on the page. If you cannot cite such a pair, the page is NOT bilingual.

Document page:
{page_text}

Return ONLY valid JSON:
{{"label": "ENGLISH" or "NON_ENGLISH" or "BILINGUAL", "reason": "cite the specific value pair if BILINGUAL, otherwise cite a representative non-English or English value"}}"""

TRANSLATE_PROMPT = """You are translating a trade or fisheries document page to English.

RULES:
1. Translate all non-English text to English, including: field values, product descriptions, species names, processing terms, units of measure, remarks, and all non-Latin script text
2. Translate non-Latin administrative terms (District, City, Street, Province, etc.)
3. For Latin-script text:
   - Proper nouns (vessel names, port names, company names, farm names, 
     person names) — preserve exactly as-is, do not translate
   - Address administrative terms (Desa, Kecamatan, Kabupaten, Kp.) — 
     preserve as-is
   - Common words with meaning that affect understanding of the product 
     (species names, product forms, processing methods, units of measure) — 
     TRANSLATE these even if in Latin script:
     * Spanish: Merluza→Hake, Entero→Whole, Congelado→Frozen, 
       Calamar→Squid, Filet sin piel→Fillet without skin,
       Filet con piel→Fillet with skin
     * Icelandic: Þorskur→Cod, Þorskbitar→Cod Chunks, kassi→box, 
       IQF-BITAR→IQF Pieces, Þorskflök→Cod Fillets
     * French: Cabillaud→Cod, Congélé→Frozen, Filet→Fillet
     * Portuguese: Lula→Squid, Congelado→Frozen, Inteiro→Whole
   - When in doubt: if the word describes what the product IS or HOW it 
     was processed, translate it. If it names a specific place, vessel, 
     or company, preserve it.
4. Return ONLY the translated document — no commentary, no explanation, no markdown fences

FORMATTING — THIS IS CRITICAL:
- Return the translated text in EXACTLY the same format as the input
- Keep all table pipes, dashes, rows, columns, blank lines, and spacing identical
- Keep all section numbers and headers in place
- Do not add, remove, or merge any rows, columns, or lines
- Do not wrap the output in code fences or quotes


{page_text}"""

In [ ]:
PIPELINE_CONFIGS = [
    # Document classification
    {
        'key':   'classify_model',
        'value': 'claude-sonnet-5',
        'desc':  'Model used for document type classification via AI_COMPLETE',
    },
    {
        'key':   'classify_prompt',
        'value': CLASSIFY_PROMPT,
        'desc':  'Prompt template for classification — {pages} is the only placeholder',
    },
    # Language check Pass 1 (Haiku — binary screen)
    {
        'key':   'lang_check_haiku_model',
        'value': 'claude-haiku-4-5',
        'desc':  'Model used for binary language screen Pass 1',
    },
    {
        'key':   'lang_check_haiku_prompt',
        'value': HAIKU_PROMPT,
        'desc':  'Haiku binary classification prompt — {page_text} is the only placeholder',
    },
    # Language check Pass 2 (Sonnet — BILINGUAL vs NON_ENGLISH)
    {
        'key':   'lang_check_sonnet_model',
        'value': 'claude-sonnet-5',
        'desc':  'Model used for BILINGUAL vs NON_ENGLISH classification Pass 2',
    },
    {
        'key':   'lang_check_sonnet_prompt',
        'value': SONNET_LANG_PROMPT,
        'desc':  'Sonnet BILINGUAL vs NON_ENGLISH prompt — {page_text} is the only placeholder',
    },
    # Translation 
    {
        'key':   'translate_model',
        'value': 'claude-sonnet-5',
        'desc':  'Model used for page translation via AI_COMPLETE',
    },
    {
        'key':   'translate_prompt',
        'value': TRANSLATE_PROMPT,
        'desc':  'Translation prompt — {page_text} is the only placeholder',
    },
    # Field extraction
    {
        'key':   'extract_model',
        'value': 'claude-sonnet-4-6',
        'desc':  'Model used for field extraction via AI_COMPLETE',
    },
]

# Upsert all configs with versioning 
for cfg in PIPELINE_CONFIGS:
    result = s.sql(f"""
        SELECT COALESCE(MAX(VERSION_NUMBER), 0) AS MAX_VERSION
        FROM PERMAFROST_POC.CONFIG.PIPELINE_CONFIG
        WHERE CONFIG_KEY = '{cfg['key']}'
    """).collect()

    max_version = result[0]['MAX_VERSION']

    # Skip if value unchanged
    if max_version > 0:
        current = s.sql(f"""
            SELECT CONFIG_VALUE
            FROM PERMAFROST_POC.CONFIG.PIPELINE_CONFIG
            WHERE CONFIG_KEY = '{cfg['key']}'
              AND IS_ACTIVE  = TRUE
        """).collect()

        if current and current[0]['CONFIG_VALUE'] == cfg['value']:
            info(f"  {cfg['key']} v{max_version} — no change, skipping")
            continue

    new_version = max_version + 1

    # Deactivate current active version
    if max_version > 0:
        s.sql(f"""
            UPDATE PERMAFROST_POC.CONFIG.PIPELINE_CONFIG
            SET IS_ACTIVE = FALSE
            WHERE CONFIG_KEY = '{cfg['key']}'
              AND IS_ACTIVE  = TRUE
        """).collect()

    # Insert new version
    s.sql(f"""
        INSERT INTO PERMAFROST_POC.CONFIG.PIPELINE_CONFIG
            (CONFIG_KEY, CONFIG_VALUE, DESCRIPTION, VERSION_NUMBER, IS_ACTIVE)
        VALUES (
            '{cfg['key']}',
            $${cfg['value']}$$,
            '{cfg['desc'].replace("'", "''")}',
            {new_version},
            TRUE
        )
    """).collect()

    info(f"  {cfg['key']} — inserted v{new_version}")

# Verify 
print("\n PIPELINE_CONFIG (active)")
s.sql("""
    SELECT
        CONFIG_KEY,
        VERSION_NUMBER,
        IS_ACTIVE,
        LEFT(CONFIG_VALUE::VARCHAR, 80) AS VALUE_PREVIEW,
        UPDATED_AT
    FROM PERMAFROST_POC.CONFIG.PIPELINE_CONFIG
    WHERE IS_ACTIVE = TRUE
    ORDER BY CONFIG_KEY
""").show()